In [ ]:
import os
import sys
sys.path.append('..')
sys.path.append('<PROJECT_ROOT_D3>/')
from Funcs.Utility import *
import numpy as np
import pandas as pd
from typing import Dict, Callable, Union, Tuple, List, Optional, Iterable
from datetime import timedelta as td
from scipy import stats
import ray
import warnings
import time
import ray
import dask

In [ ]:
print(PATH_INTERMEDIATE)

In [ ]:
esm_df = pd.read_csv(PATH_ESM)

In [ ]:
esm_df

In [ ]:
esm_df = esm_df[['pcode', 'responseTime', 'stress']]

In [ ]:
esm_df

In [ ]:
# Convert responseTime and stress to numeric, coerce errors to NaN
esm_df['responseTime'] = pd.to_numeric(esm_df['responseTime'], errors='coerce')
esm_df['stress'] = pd.to_numeric(esm_df['stress'], errors='coerce')

# Drop rows with NaN values
esm_df.dropna(subset=['responseTime', 'stress'], inplace=True)

In [ ]:
print("NaN values per column:")
print(esm_df.isna().sum())

In [ ]:
# Check the unique values of responseTime and stress
print("Unique values in responseTime:", esm_df['responseTime'].unique())
print("Unique values in stress:", esm_df['stress'].unique())

In [ ]:
esm_df.head()

In [ ]:
print("Basic Statistics:")
print(esm_df.describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(esm_df['responseTime'], bins=30)
plt.title('Distribution of Response Time')

plt.subplot(1, 2, 2)
plt.hist(esm_df['stress'], bins=30)
plt.title('Distribution of Stress')

plt.tight_layout()
plt.show()

In [ ]:
# def calculate_step_count(user_folder, response_time):
#     step_file_path = os.path.join(user_folder, 'Fitbit-Calorie.csv')
#     if not os.path.exists(step_file_path):
#         print(f"File does not exist: {step_file_path}")
#         return None

#     steps_df = pd.read_csv(step_file_path)
#     steps_df['timestamp'] = pd.to_numeric(steps_df['timestamp'], errors='coerce')
#     steps_df['value'] = pd.to_numeric(steps_df['value'], errors='coerce')

#     start_time = response_time - 30 * 60 * 1000  # 30 minutes before
#     end_time = response_time + 30 * 60 * 1000   # 30 minutes after

#     # Filter steps within the 1-hour window
#     mask = (steps_df['timestamp'] >= start_time) & (steps_df['timestamp'] <= end_time)
#     step_count = steps_df.loc[mask, 'value'].sum()

#     print(f"User folder: {user_folder}, Response time: {response_time}, Calorie count: {step_count}")

#     return step_count

In [ ]:
# step_counts = []
# for index, row in esm_df.iterrows():
#     user_folder = os.path.join(PATH_SENSOR, row['pcode'])
#     response_time = row['responseTime']
#     step_count = calculate_step_count(user_folder, response_time)
#     step_counts.append(step_count)

In [ ]:
# esm_df['calorie_count'] = step_counts

In [ ]:
print(esm_df)

In [ ]:
def remove_first_last(df_grouped):
    return df_grouped.iloc[1:-1]

esm_df = esm_df.groupby('pcode').apply(remove_first_last).reset_index(drop=True)

In [ ]:
# esm_df = esm_df[['pcode', 'responseTime', 'stress', 'calorie_count']].copy()

# # Convert the 'responseTime' column to datetime format without using .loc
# esm_df['responseTime'] = pd.to_datetime(esm_df['responseTime'], unit='ms', utc=True).dt.tz_convert(DEFAULT_TZ)

# # Display the DataFrame to verify the changes
# print(esm_df.head())
esm_df = esm_df[['pcode', 'responseTime', 'stress']].copy()

# Convert the 'responseTime' column to datetime format without using .loc
esm_df['responseTime'] = pd.to_datetime(esm_df['responseTime'], unit='ms', utc=True).dt.tz_convert(DEFAULT_TZ)

# Display the DataFrame to verify the changes
print(esm_df.head())

In [ ]:
esm_df = esm_df.rename(columns={'responseTime': 'timestamp'})

In [ ]:
esm_df 

In [ ]:
# esm_df.to_csv(os.path.join(PATH_INTERMEDIATE, 'labels_1h_esmsyn.csv'), index=False)

In [ ]:
# esm_path = os.path.join('Intermediate', 'labels_1h_esmsyn.csv')
# # df = pd.read_csv(esm_path)

In [ ]:
df = esm_df

In [ ]:
df

In [ ]:
print("Basic Statistics:")
print(df.describe())

In [ ]:
# Simple Matplotlib histograms for distribution plots
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(df['stress'], bins=30)
plt.title('Distribution of Stress')

# plt.subplot(1, 2, 2)
# plt.hist(df['calorie_count'], bins=30)
# plt.title('Distribution of Calorie Count')

plt.tight_layout()
plt.show()

In [ ]:
# zero_count = (df['calorie_count'] == 0).sum()
# print(zero_count)

Binarization

In [ ]:
# # Calculate personal means
# personal_means = df.groupby('pcode').mean()[['stress', 'calorie_count']]

# # Calculate overall means
# overall_mean_stress = df['stress'].mean()
# overall_mean_calories = df['calorie_count'].mean()

# print("Personal Means:")
# print(personal_means.head())

# print("Overall Mean Stress:", overall_mean_stress)
# print("Overall Mean Calorie:", overall_mean_calories)

# Calculate personal means
personal_means = df.groupby('pcode').mean()[['stress']]

# Calculate overall means
overall_mean_stress = df['stress'].mean()

print("Personal Means:")
print(personal_means.head())

print("Overall Mean Stress:", overall_mean_stress)


In [ ]:
# # Function to binarize based on personal means
# def binarize_personal(row, personal_means):
#     pcode = row['pcode']
#     personal_mean_stress = personal_means.loc[pcode, 'stress']
#     personal_mean_calories = personal_means.loc[pcode, 'calorie_count']
    
#     row['stress_binary_personal'] = 1 if row['stress'] > personal_mean_stress else 0
#     row['calorie_count_binary_personal'] = 1 if row['calorie_count'] > personal_mean_calories else 0
#     return row

# # Binarize based on personal means
# df = df.apply(binarize_personal, axis=1, personal_means=personal_means)

# # Binarize based on overall means
# df['stress_binary_overall'] = df['stress'].apply(lambda x: 1 if x > overall_mean_stress else 0)
# df['calorie_count_binary_overall'] = df['calorie_count'].apply(lambda x: 1 if x > overall_mean_calories else 0)

# # Display the updated DataFrame
# print(df.head())

# Function to binarize based on personal means
def binarize_personal(row, personal_means):
    pcode = row['pcode']
    personal_mean_stress = personal_means.loc[pcode, 'stress']
    # personal_mean_calories = personal_means.loc[pcode, 'calorie_count']
    
    row['stress_binary_personal'] = 1 if row['stress'] > personal_mean_stress else 0
    # row['calorie_count_binary_personal'] = 1 if row['calorie_count'] > personal_mean_calories else 0
    return row

# Binarize based on personal means
df = df.apply(binarize_personal, axis=1, personal_means=personal_means)

# Binarize based on overall means
df['stress_binary_overall'] = df['stress'].apply(lambda x: 1 if x > overall_mean_stress else 0)
# df['calorie_count_binary_overall'] = df['calorie_count'].apply(lambda x: 1 if x > overall_mean_calories else 0)

# Display the updated DataFrame
print(df.head())


In [ ]:
df

In [ ]:
df = df.drop_duplicates(subset=['pcode', 'timestamp'])

In [ ]:
df

In [ ]:
esm_path = os.path.join(PATH_INTERMEDIATE, 'labels_1h_esmsyn.csv')
df.to_csv(esm_path, index=False)

In [ ]:
# # EDA for Binarized Data

# # Distribution plots for personal binarized data using Matplotlib
# plt.figure(figsize=(12, 6))

# plt.subplot(1, 2, 1)
# plt.hist(df['stress_binary_personal'], bins=2, edgecolor='k')
# plt.title('Binarized Stress (Personal Mean)')

# plt.subplot(1, 2, 2)
# plt.hist(df['calorie_count_binary_personal'], bins=2, edgecolor='k')
# plt.title('Binarized Calorie Count (Personal Mean)')

# plt.tight_layout()
# plt.show()

# # Distribution plots for overall binarized data using Matplotlib
# plt.figure(figsize=(12, 6))

# plt.subplot(1, 2, 1)
# plt.hist(df['stress_binary_overall'], bins=2, edgecolor='k')
# plt.title('Binarized Stress (Overall Mean)')

# plt.subplot(1, 2, 2)
# plt.hist(df['calorie_count_binary_overall'], bins=2, edgecolor='k')
# plt.title('Binarized Calorie Count (Overall Mean)')

# plt.tight_layout()
# plt.show()

# # Correlation analysis for binarized data
# correlation_binarized = df[['stress_binary_personal', 'calorie_count_binary_personal',
#                             'stress_binary_overall', 'calorie_count_binary_overall']].corr()
# print("Correlation Matrix (Binarized Data):")
# print(correlation_binarized)

# # Heatmap for correlation matrix (binarized data) using Seaborn
# plt.figure(figsize=(8, 6))
# sns.heatmap(correlation_binarized, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
# plt.title('Correlation Matrix (Binarized Data)')
# plt.show()

In [ ]:
# zero_count = (df['calorie_count_binary_personal'] == 0).sum()
# print(zero_count)

In [ ]:
# zero_count = (df['calorie_count_binary_overall'] == 0).sum()
# print(zero_count)